pydantic library

In [9]:
from pydantic import BaseModel, Field

class User(BaseModel):
    name: str = Field(min_length=2)
    age: int = Field(ge=0)  # ge=0 means greater than or equal to 0
    email: str

# 1. Parsing and validating raw JSON/dict data:
raw_input = {"name": "Alice", "age": "25", "email": "alice@example.com"}
user = User(**raw_input)

print(user.age)        # 25 (automatically converted from string to int)
print(type(user.age))  # <class 'int'>

# 2. Exporting back to dict or JSON:
user_dict = user.model_dump()       # {'name': 'Alice', 'age': 25, 'email': 'alice@example.com'}
user_json = user.model_dump_json()  # '{"name":"Alice","age":25,"email":"alice@example.com"}'

25
<class 'int'>


In [ ]:
"""OpenAI Structured Outputs backend for paper evidence extraction."""

from __future__ import annotations

from typing import Any, Sequence

from pydantic import BaseModel, ConfigDict, Field, ValidationError

from src.config import openai_api_key, openai_extraction_model
from src.models.paper import Paper

from src.extraction.evidence import EvidenceItem, LimitationEvidence, PaperEvidence


class PaperExtractionError(RuntimeError):
    """Raised when a paper cannot be converted into structured evidence."""


class _Claim(BaseModel) :
    model_config = ConfigDict(extra = "forbid", strict = True, str_strip_whitespace = True)
    value : str = Field(min_length = 1)
    evidence_text : str | None = None
    source : str
    confidence : float = Field(ge = 0.0, le = 1.0)


class _Limitation(_Claim) :
    author_stated : bool


class _ExtractionResult(BaseModel) :
    model_config = ConfigDict(extra = "forbid", strict = True)
    research_objective = _Claim | None = None
    population_or_setting : list[_Claim] = Field(default_factory = list)
    method_or_intervention: list[_Claim] = Field(default_factory=list)
    comparison_or_baseline: list[_Claim] = Field(default_factory=list)
    datasets: list[_Claim] = Field(default_factory=list)
    sample_size: _Claim | None = None
    evaluation_metrics: list[_Claim] = Field(default_factory=list)
    main_findings: list[_Claim] = Field(default_factory=list)
    limitations: list[_Limitation] = Field(default_factory=list)
    future_work: list[_Claim] = Field(default_factory=list)
    extraction_confidence: float = Field(ge=0.0, le=1.0)


_INSTRUCTIONS = """
Extract structured evidence only from the supplied paper title and abstract.
Do not use outside knowledge or infer missing experimental details. Do not
invent datasets, sample sizes, baselines, metrics, findings, limitations, or
future work. Return null or an empty list when information is absent.

Every claim must include supporting text copied from the title or abstract and
source must be exactly title or abstract. A limitation may be author_stated
only when the text explicitly presents it as a limitation, weakness,
constraint, shortcoming, or equivalent. Generic criticism is not a limitation.
Future work must be an explicit direction proposed by the authors. Sample size
must be explicit. Keep claims concise and preserve stated numerical results.
""".strip()

class PaperExtractor :
    def __init__(self, *, client, api_key : str, 
                 model : str, max_output_tokens : int, evidence_limit : int = 10) -> None :
        if max_output_tokens <= 0 or evidence_limit < 0 :
            raise()
        self.model = model or openai_extraction_model()
        self.max_output_tokens = max_output_tokens
        self.evidence_limit = evidence_limit
        self.failures : list[PaperExtractionError] = []
        if client is not None :
            self.client = client
            return 
        key = api_key or openai_api_key()
        if not key :
            raise PaperExtractionError("")
        key = api_key or openai_api_key()
        if not key :
            raise PaperExtractionError()
        from openai import OpenAI
        self.client = OpenAI(api_key = key)
    
    def extract(self, paper : Paper) -> PaperEvidence :
        title = paper.title.strip()
        abstract = paper.abstract.strip() if paper.abstract else None
        if not title :
            raise PaperExtractionError()
        source = f""
        try :
            response = self.client.response.parse(
                model = self.model, reasoning = {"effort" : "low"}, store = False, 
                max_output_tokens = self.max_output_tokens, instructions = _INSTRUCTIONS, 
                input = source, text_format = _ExtractionResult
            )
            payload = getattr(response, "output_parsed", None)
            return _to_evidence(paper, payload)
        except PaperExtractionError :
            raise

    def extract_many(self, papers : Sequence[Paper], limit : int | None = None) :
        self.failures = []
        if limit is not None and limit < 0 :
            raise ValueError("limit must be non-negative.")
        selected = list(papers)[:self.evidence_limit if limit is None else limit]
        results : list[PaperEvidence] = []
        for paper in selected :
            try :
                results.append(self.extract(paper))
            except PaperExtractionError as exc :
                self.failures.append({exc})
        return results
    
def _claim(value : _Claim) -> EvidenceItem :
    return EvidenceItem(value = value.value, evidence_text = value.evidence_text)


def _to_evidence(paper : Paper, payload : _ExtractionResult) -> PaperEvidence :
    def claims(values : list[_Claim]) -> list[EvidenceItem] :
        return [(_claim(item) for item in values if item.source in ("title", "abstract"))]
    limitations = [
        LimitationEvidence(**_claim(item).model_dump(), author_state = True)
        for item in payload.limitations if item.author_stated and item.source in ("title", "abstract")
    ]

    return PaperEvidence(
        paper_id = paper.id, title = paper.title, 
        research_objective = claims(payload.research_objective) if payload.research_objective and payload.research_objective in ("title", "abstract") else None
        comparison_or_baseline = claims(payload.comparison_or_baseline),
    )

